# Part Alternative Risk Score (PARS)
### A defensible, ISO 31000 / IEC 62402 / ASCM-aligned score for `Parts_Alternative_rows.csv`

**Scope.** This notebook builds a standalone risk score for the *Alternative* relationship
itself — i.e. **"if I have to rely on this documented alternate part, how risky is that
substitution?"** — as distinct from the broader multi-table `Parts_Risk_Assessment_.ipynb`
model, which uses the Alternative table only as *one input* into an overall part-obsolescence
score (`N_SSE`, `N_AQR`, `N_MDR` feeding into `CRS`).

**Why a separate notebook, not an edit to the existing one.** The existing notebook answers
*"how exposed is this primary part, considering obsolescence, compliance, packaging, docs, and
alternates together?"* This notebook answers a narrower, different question — *"how good/risky
is this specific alternate-part record?"* — using only the single table provided
(`Parts_Alternative_rows.csv`), with no dependency on `Parts.xlsx`, `ROH.xlsx`, or
`Manufactor.xlsx`. The methodology is reviewed and selectively reused (Section 3) rather than
copied wholesale, because several of its normalization steps do not map cleanly onto a
row-level "alternative risk" target.

**Standards referenced:**
- **ISO 31000:2018** — risk = function of likelihood and consequence; qualitative-to-quantitative
  risk matrices; risk treatment via redundancy/diversification.
- **IEC 62402:2019 (Obsolescence Management)** — Form-Fit-Function (FFF) change categories,
  which map directly onto this dataset's `relationship` values.
- **ASCM/APICS SCRM body of knowledge** — single-sourcing as a structural supply risk driver,
  multi-sourcing as a recognized risk-treatment/mitigation.
- **Sandborn (DMSMS / obsolescence cost) literature** — single-source exposure as the dominant
  cost driver when a part cannot be resupplied.


In [1]:
# CELL 1: Imports & load
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)

df = pd.read_csv('Parts_Alternative_rows.csv')
print(f"Shape: {df.shape}")
df.head()


Shape: (568, 7)


,id,Mfr Part #,Alternative Part,manufacturer,relationship,description,Last_Modified_Date
0,1,ADA44301YKSZR7,ADA44301YKSZR,Analog Devices,Identical,"Same silicon/die, different packaging or order...",2026-07-14 13:37:41.633463+00
1,2,EQCO30T5.2,EQCO30T5.3,Semtech,Similar,"Same function or product family, but requires ...",2026-07-14 13:37:41.633463+00
2,3,SN65DPHY440SSRHRR,SN65DPHY440SSRHR,Texas Instruments,Identical,"Same silicon/die, different packaging or order...",2026-07-14 13:37:41.633463+00
3,4,EQCO30R5.D,EQCO30R5.2,Semtech,Similar,"Same function or product family, but requires ...",2026-07-14 13:37:41.633463+00
4,5,AD8195ACPZR7,AD8195ACPZR,Analog Devices,Identical,"Same silicon/die, different packaging or order...",2026-07-14 13:37:41.633463+00


## 1. Feature Analysis — every column in `Parts_Alternative_rows.csv`

| Column | What it is | Include in score? | Why |
|---|---|---|---|
| `id` | Row surrogate key | ❌ No | Arbitrary identifier, carries no risk information. |
| `Mfr Part #` | The original/primary part number | ⚠️ Structural only | Used as the **join/grouping key** to roll row-level scores up to a part-level score (Section 6), not itself a risk input. |
| `Alternative Part` | The candidate substitute part number | ⚠️ Structural + diagnostic | Used to detect the literal string `"No Alternative"` sentinel and to compute a diagnostic part-number-divergence metric (Section 3.2) — tested and **excluded** from the final score after empirical review. |
| `manufacturer` | Manufacturer of the **alternative** part | ❌ No independent signal | Fully collinear with `relationship`: every row where `manufacturer` is missing is *exactly* the set of rows where `relationship` is missing (verified in Section 2). It carries zero information not already in `relationship`. (It *would* be informative for a portfolio-level "alternate-manufacturer diversity" metric, as used in the reference notebook's `N_MDR` — but that measures resilience of the *primary part's sourcing*, not the risk of *this specific alternative*, so it's out of scope for a row-level Alternative Risk Score.) |
| `relationship` | `Identical` / `Direct` / `Similar` / missing | ✅ **Yes — primary driver** | This is the only field in the table that is a direct, engineering-reviewed statement of substitution risk (see Section 3.1). Higher-risk order: `Similar` > `Direct` > `Identical`; a missing/blank record (in practice, `Alternative Part = "No Alternative"`) is the highest-risk case of all — no documented substitution path exists. |
| `description` | Free text | ❌ Redundant | Verified 1:1 with `relationship` — there are exactly 4 distinct description strings, one per `relationship` value, with zero variation within a category (Section 2). Carries no incremental signal; retained only as a human-readable label in outputs. |
| `Last_Modified_Date` | Row timestamp | ❌ No signal | Every row shares the **identical** timestamp (Section 2) — a single bulk-load timestamp, not a per-record last-verified date. With zero variance it cannot support a data-recency/staleness risk factor, however desirable that would be in principle. |

**Direction of risk (stated explicitly, per instructions not to leave it implicit):**
- `relationship` moving from `Identical → Direct → Similar → (none)` **increases** risk (looser
  substitution match, more required engineering validation, or no substitute at all).
- Number of *real* (non-placeholder) alternative rows per `Mfr Part #` — **decreases** risk as it
  increases (more redundancy / sourcing options), consistent with ASCM multi-sourcing guidance and
  the reference notebook's `N_SSE`.


In [2]:
# CELL 2: Confirm the collinearity / redundancy claims made above

# (a) manufacturer missing <=> relationship missing?
rel_missing_mask = df['relationship'].isna() | df['relationship'].str.contains('€', na=False)
mfr_missing_mask = df['manufacturer'].isna() | df['manufacturer'].str.contains('€', na=False)
print("manufacturer-missing rows == relationship-missing rows:",
      (rel_missing_mask == mfr_missing_mask).all(),
      f"({rel_missing_mask.sum()} rows)")

# (b) description is a deterministic function of relationship?
print("\nUnique (relationship -> description) mappings:")
print(df.groupby('relationship')['description'].nunique())

# (c) Last_Modified_Date has zero variance?
print("\nUnique Last_Modified_Date values:", df['Last_Modified_Date'].nunique())


manufacturer-missing rows == relationship-missing rows: True (62 rows)

Unique (relationship -> description) mappings:
relationship
Direct       1
Identical    1
Similar      1
â€”          1
Name: description, dtype: int64

Unique Last_Modified_Date values: 1


## 2. Data Cleaning

Two genuine data-quality issues were found (both also present in the source tables behind the
reference notebook, confirming this is an upstream export artifact, not specific to this file):

1. **Mojibake.** `manufacturer`, `relationship`, and `description` show UTF‑8 text that was
   re-encoded through **Windows‑1252** (classic "em dash becomes `â€”`" corruption). Fixed by
   round-tripping through `cp1252 → utf-8`, the same technique the reference notebook uses.
2. **One self-referencing row** (`Mfr Part # == Alternative Part`, e.g. `MPC8270CZQMIBA →
   MPC8270CZQMIBA`, tagged `Identical`). A part cannot be its own alternative — this is a
   data-entry error, not a real "true drop-in" relationship. Consistent with the reference
   notebook's Section 13 fix, this row is corrected: it is **excluded from counting as a real
   alternative** for that part, but not deleted from the table (so it remains visible for audit).


In [3]:
# CELL 3: Clean the data
def fix_mojibake(val):
    if not isinstance(val, str):
        return val
    if any(ch in val for ch in ['â', 'Ã', '€']):
        try:
            return val.encode('cp1252').decode('utf-8')
        except (UnicodeDecodeError, UnicodeEncodeError):
            return val
    return val

clean = df.copy()
for c in ['manufacturer', 'relationship', 'description']:
    clean[c] = clean[c].apply(fix_mojibake)

for c in ['Mfr Part #', 'Alternative Part']:
    clean[c] = clean[c].astype(str).str.strip()

print("Relationship values after cleanup:", clean['relationship'].unique().tolist())

# Flag the two special cases
clean['is_no_alternative'] = clean['Alternative Part'].str.strip().str.lower().eq('no alternative')
clean['is_self_reference'] = (
    clean['Mfr Part #'].str.upper() == clean['Alternative Part'].str.upper()
) & (~clean['is_no_alternative'])

print(f"'No Alternative' placeholder rows: {clean['is_no_alternative'].sum()}")
print(f"Self-referencing data-entry errors: {clean['is_self_reference'].sum()}")
clean.loc[clean['is_self_reference'], ['Mfr Part #', 'Alternative Part', 'manufacturer', 'relationship']]


Relationship values after cleanup: ['Identical', 'Similar', '—', 'Direct']
'No Alternative' placeholder rows: 62
Self-referencing data-entry errors: 1


,Mfr Part #,Alternative Part,manufacturer,relationship
161,MPC8270CZQMIBA,MPC8270CZQMIBA,NXP Semiconductors,Identical


## 3. Review of the Reference Notebook's Methodology

The reference notebook (`Parts_Risk_Assessment_.ipynb`) already uses the Alternative table to
derive three factors, all folded into its **Impact** axis of an overall part-obsolescence score:

| Reference notebook factor | Formula | What it actually measures |
|---|---|---|
| `N_SSE` (single-source exposure) | `100 / (1 + alt_count)` | Whether *any* real alternative exists, and how many — quantity only. |
| `N_AQR` (alternate quality) | `(1 − best_rel_rank/3) × 100` | The quality of the *best* documented alternative — quality only, per part. |
| `N_MDR` (alt-manufacturer diversity) | based on `nunique(Manufacturer)` among alternates | Whether the primary part's *sourcing* is diversified across vendors. |

### 3.1 What's reusable, and why

`N_SSE`'s core idea — **risk falls as the count of real alternatives rises, with the sharpest
drop from 0→1** — is a well-established DMSMS/Sandborn principle and is **reused here** (Section
4.2) as the "redundancy credit" applied at the part level. `N_AQR`'s core idea — **rank
`Identical > Direct > Similar > none`** — is also correct in direction and is **reused** as the
ordinal basis for this notebook's row-level score, re-derived independently below from the
relationship *descriptions themselves* (IEC 62402 Form-Fit-Function reasoning) rather than copied
as a bare 3-2-1-0 rank, so the anchor values are explainable on their own terms.

### 3.2 What's *not* reused, and why

- **`N_MDR` (manufacturer diversity) does not apply here.** It answers "is the primary part's
  supply diversified across vendors?" — a part-portfolio question. This notebook's target is
  "how risky is *this one* alternative record?", for which the alternate's manufacturer name adds
  no information beyond what `relationship` already encodes (Section 1/2).
- **A part-number-string-similarity feature was tested and rejected.** It is intuitive to guess
  that alternates with near-identical part numbers (e.g. differing only by a packaging suffix)
  represent lower engineering risk than alternates with completely different part numbers. This
  was tested directly (normalized Levenshtein edit distance between `Mfr Part #` and `Alternative
  Part`) and **the data contradicts the assumption**: `Direct` relationships (fully
  interchangeable, no design change) show the *highest* mean divergence (≈0.53), higher than
  `Similar` (≈0.23) or `Identical` (≈0.14) — because a `Direct` alternate is frequently a
  cross-manufacturer second source with its own independent numbering scheme, while `Identical`
  alternates are almost always the *same* manufacturer's own repackaging code. String similarity
  therefore measures "same naming family," not "engineering closeness" — and using it as a risk
  modifier would have *penalized* genuinely good cross-vendor second sources for looking
  different. It is shown for transparency in Section 3.3 and then dropped, per the instruction not
  to add complexity without demonstrated benefit.

This is the intended outcome of "review before reusing": two of the reference notebook's three
Alternative-derived ideas transfer with re-justification; one does not apply at this grain; and
one candidate feature of my own was tested and discarded on the evidence.


In [4]:
# CELL 4: Diagnostic — test the part-number-similarity hypothesis (kept for transparency, not used in scoring)
def levenshtein(a, b):
    a, b = str(a).upper().strip(), str(b).upper().strip()
    n, m = len(a), len(b)
    if n == 0: return m
    if m == 0: return n
    prev = list(range(m + 1))
    for i in range(1, n + 1):
        cur = [i] + [0] * m
        for j in range(1, m + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + cost)
        prev = cur
    return prev[m]

def divergence_ratio(a, b):
    a, b = str(a).upper().strip(), str(b).upper().strip()
    denom = max(len(a), len(b), 1)
    return levenshtein(a, b) / denom

real = clean[~clean['is_no_alternative']].copy()
real['divergence'] = real.apply(
    lambda r: divergence_ratio(r['Mfr Part #'], r['Alternative Part']), axis=1
)
print("Mean part-number divergence by relationship (lower = more similar strings):")
print(real.groupby('relationship')['divergence'].mean().sort_values())
print("\n-> Not monotonic with expected engineering risk order (Identical < Direct < Similar).")
print("-> Confirms the decision in Section 3.2 to exclude this as a scoring feature.")


Mean part-number divergence by relationship (lower = more similar strings):
relationship
Identical    0.136333
Similar      0.233130
Direct       0.526461
Name: divergence, dtype: float64

-> Not monotonic with expected engineering risk order (Identical < Direct < Similar).
-> Confirms the decision in Section 3.2 to exclude this as a scoring feature.


## 4. Scoring Methodology

### 4.1 Row-level: Alternative Substitution Risk (`ASR`)

Each row's `relationship` value is mapped to a fixed anchor on 0–100, derived directly from what
the relationship *actually claims* about engineering-validation effort (IEC 62402 FFF categories),
not an arbitrary ordinal spacing:

| `relationship` | What it claims | Engineering-validation effort implied | `ASR` |
|---|---|---|---|
| `Identical` | Same silicon/die, packaging/order-code difference only | None — no engineering review needed (per the data's own description) | **5** |
| `Direct` | Same pinout/package/key electrical specs, fully interchangeable | None claimed, but usually a different manufacturer/die — residual, unverified parametric risk (thermal, tolerance, qualification grade) that a pinout/spec match alone doesn't rule out | **25** |
| `Similar` | Same function/family only | Real: explicit datasheet comparison required before use; possible board/firmware impact | **65** |
| *(none — "No Alternative")* | No substitute documented at all | N/A — there is nothing to validate against | **100** |

A small residual score (5, not 0) is kept for `Identical` because even a same-silicon repackage
still requires an ERP/BOM part-number change and carries non-zero logistics/date-code risk — a
score of exactly 0 would overstate certainty. `100` for "no alternative" is a **fixed value, not
an estimate** — it mirrors the reference notebook's own DMSMS hard-stop treatment of zero real
alternatives, which is appropriate to reuse here since it is the same underlying condition.

These four anchors are the *entire* row-level model. No other row-level feature survived Section
3's review, so there is nothing left to weight — avoiding exactly the kind of arbitrary
multi-factor weighting the brief warns against.

### 4.2 Part-level: Part Alternative Risk Score (`PARS`)

A part with two mediocre alternatives is not treated as risky as a part with one, because ASCM/ISO
31000 both recognize multi-sourcing as a genuine (if partial) risk treatment. `PARS` for a given
`Mfr Part #` p is:

$$
PARS(p) = \begin{cases}
100 & \text{if } p \text{ has zero real alternatives (hard stop)} \\[4pt]
\min_{r \,\in\, \text{real alternatives of } p} ASR(r) \;\times\; \rho(n_p) & \text{otherwise}
\end{cases}
$$

where $n_p$ is the count of real (non-placeholder, non-self-referencing) alternatives for part
$p$, and $\rho$ is a **bounded redundancy credit**:

| $n_p$ (real alt. count) | $\rho(n_p)$ | Justification |
|---|---|---|
| 1 | 1.00 | Single point of failure — if this one alternative turns out unusable on validation, there is no fallback. |
| 2 | 0.90 | A second independent option exists; modest (10%) credit. |
| ≥3 | 0.80 | Diminishing returns — capped at 20% credit, since the *best* alternative's own quality still dominates the real-world substitution risk; extra low-quality options don't meaningfully de-risk a bad best-case. |

The credit is **multiplicative and capped**, not a linear weighted sum, precisely so it can never
let quantity fully override quality (e.g. three `Similar` alternatives cannot out-score one
genuine `Identical` alternative) — anchoring the score on the single best documented path, with
count only providing bounded, secondary mitigation credit. This keeps the model to two,
independently-justified parameters (`ASR` anchors, `ρ` credits) rather than an AHP-style weight
search across many correlated inputs, appropriate given only one substantive underlying feature
(`relationship`) is available in this dataset.

### 4.3 Risk classification (0–100 → 5-level ISO risk-matrix labels)

| Score range | Category |
|---|---|
| 0 – 19 | Very Low |
| 20 – 39 | Low |
| 40 – 59 | Medium |
| 60 – 79 | High |
| 80 – 100 | Critical |


In [5]:
# CELL 5: Implementation

ASR_MAP = {'Identical': 5, 'Direct': 25, 'Similar': 65}  # 'No Alternative' handled separately

def classify(score):
    if score < 20:  return 'Very Low'
    if score < 40:  return 'Low'
    if score < 60:  return 'Medium'
    if score < 80:  return 'High'
    return 'Critical'

# --- Row-level ASR ---
scored = clean.copy()
scored['ASR'] = np.where(
    scored['is_no_alternative'], 100.0,
    scored['relationship'].map(ASR_MAP)
)
# Missing/unmapped relationship values (defensive; none expected after cleanup) -> treat as unknown = max risk
scored['ASR'] = scored['ASR'].fillna(100.0)
scored['ASR_category'] = scored['ASR'].apply(classify)

print(scored['ASR'].isna().sum(), "rows with unresolved ASR (should be 0)")
scored[['id', 'Mfr Part #', 'Alternative Part', 'relationship', 'ASR', 'ASR_category']].head(10)


0 rows with unresolved ASR (should be 0)


,id,Mfr Part #,Alternative Part,relationship,ASR,ASR_category
0,1,ADA44301YKSZR7,ADA44301YKSZR,Identical,5.0,Very Low
1,2,EQCO30T5.2,EQCO30T5.3,Similar,65.0,High
2,3,SN65DPHY440SSRHRR,SN65DPHY440SSRHR,Identical,5.0,Very Low
3,4,EQCO30R5.D,EQCO30R5.2,Similar,65.0,High
4,5,AD8195ACPZR7,AD8195ACPZR,Identical,5.0,Very Low
5,6,FT800QR,FT800QL,Identical,5.0,Very Low
6,7,MAX9406ETJ+,MAX9406ETJ+T,Identical,5.0,Very Low
7,8,MAX4886ETO+,MAX4886ETO+T,Identical,5.0,Very Low
8,9,LMH1980MM/NOPB,LMH1980MME/NOPB,Identical,5.0,Very Low
9,10,SN65DP159RGZR,SN65DP159RGZT,Identical,5.0,Very Low


In [6]:
# CELL 6: Part-level rollup -> PARS

RHO = {1: 1.00, 2: 0.90}  # n>=3 handled via .get(..., 0.80)

def rho(n):
    return RHO.get(n, 0.80)

real_alts = scored[(~scored['is_no_alternative']) & (~scored['is_self_reference'])].copy()

part_agg = (
    real_alts.groupby('Mfr Part #')
    .agg(best_ASR=('ASR', 'min'), n_real_alts=('ASR', 'count'))
    .reset_index()
)
part_agg['redundancy_credit'] = part_agg['n_real_alts'].apply(rho)
part_agg['PARS'] = (part_agg['best_ASR'] * part_agg['redundancy_credit']).round(1)

# Parts with ZERO real alternatives (either explicit "No Alternative" rows, or only a
# self-referencing row and nothing else) get the fixed hard-stop score.
all_parts = pd.DataFrame({'Mfr Part #': clean['Mfr Part #'].unique()})
part_scores = all_parts.merge(part_agg, on='Mfr Part #', how='left')
no_real_alt_mask = part_scores['PARS'].isna()
part_scores.loc[no_real_alt_mask, ['best_ASR', 'n_real_alts', 'redundancy_credit']] = [100.0, 0, 1.0]
part_scores.loc[no_real_alt_mask, 'PARS'] = 100.0
part_scores['PARS_category'] = part_scores['PARS'].apply(classify)

print(f"Parts scored: {len(part_scores)}")
print(f"  Zero real alternatives (hard stop, PARS=100): {no_real_alt_mask.sum()}")
part_scores['PARS_category'].value_counts().reindex(
    ['Very Low', 'Low', 'Medium', 'High', 'Critical']
).fillna(0).astype(int)


Parts scored: 433
  Zero real alternatives (hard stop, PARS=100): 62


PARS_category
Very Low    246
Low          25
Medium       11
High         89
Critical     62
Name: count, dtype: int32

## 5. Validation — does the score behave logically?

Five checks below, each targeting a specific claim the methodology makes.


In [7]:
# CELL 7: Validation 1 — full spectrum sample (one part per category)
sample_rows = []
for cat in ['Very Low', 'Low', 'Medium', 'High', 'Critical']:
    subset = part_scores[part_scores['PARS_category'] == cat]
    if len(subset):
        sample_rows.append(subset.sort_values('PARS').iloc[len(subset)//2])

pd.DataFrame(sample_rows)[['Mfr Part #', 'best_ASR', 'n_real_alts', 'redundancy_credit', 'PARS', 'PARS_category']]


,Mfr Part #,best_ASR,n_real_alts,redundancy_credit,PARS,PARS_category
37,GS2984INTE3,5.0,1.0,1.0,5.0,Very Low
190,CAT9555YIT2,25.0,2.0,0.9,22.5,Low
271,LM1972MX/NOPB,65.0,2.0,0.9,58.5,Medium
208,XRA1201PIG24TRF,65.0,1.0,1.0,65.0,High
232,TAS5010IPFBR,100.0,0.0,1.0,100.0,Critical


In [8]:
# CELL 8: Validation 2 — multi-sourcing credit lowers risk, but never below the best alternative's own tier boundary
multi = part_scores[part_scores['n_real_alts'] >= 2].sort_values('PARS').head(8)
print("Parts with 2+ real alternatives -- redundancy credit visibly reduces PARS vs best_ASR alone:")
multi[['Mfr Part #', 'best_ASR', 'n_real_alts', 'redundancy_credit', 'PARS', 'PARS_category']]


Parts with 2+ real alternatives -- redundancy credit visibly reduces PARS vs best_ASR alone:


,Mfr Part #,best_ASR,n_real_alts,redundancy_credit,PARS,PARS_category
10,FT810QR,5.0,2.0,0.9,4.5,Very Low
350,ADN2812ACPZ,5.0,2.0,0.9,4.5,Very Low
316,LM386MM1,5.0,2.0,0.9,4.5,Very Low
304,TPA3001D1PWPR,5.0,2.0,0.9,4.5,Very Low
302,SSM2211SZ,5.0,2.0,0.9,4.5,Very Low
301,NJM2113D,5.0,2.0,0.9,4.5,Very Low
300,PAM8403DRH,5.0,2.0,0.9,4.5,Very Low
294,TPA3118D2DAPR,5.0,2.0,0.9,4.5,Very Low


In [9]:
# CELL 9: Validation 3 -- hard-stop parts (no documented alternative) are all correctly Critical
hard_stop = part_scores[part_scores['n_real_alts'] == 0]
print(f"{len(hard_stop)} parts with zero real alternatives.")
print("All classified Critical:", (hard_stop['PARS_category'] == 'Critical').all())
hard_stop.head(5)


62 parts with zero real alternatives.
All classified Critical: True


,Mfr Part #,best_ASR,n_real_alts,redundancy_credit,PARS,PARS_category
14,AP1302CSSL00SMGA0DR,100.0,0.0,1.0,100.0,Critical
21,EQCO62X20C1I/3DW,100.0,0.0,1.0,100.0,Critical
50,CAP200DGTL,100.0,0.0,1.0,100.0,Critical
56,PAM8904JER,100.0,0.0,1.0,100.0,Critical
67,A5000R2HQ1/Z016UZ,100.0,0.0,1.0,100.0,Critical


In [10]:
# CELL 10: Validation 4 -- the self-referencing data-entry error does not falsely lower risk
self_ref_part = clean.loc[clean['is_self_reference'], 'Mfr Part #'].iloc[0]
print(f"Part with the self-referencing row: {self_ref_part}")
print(part_scores[part_scores['Mfr Part #'] == self_ref_part])
print()
print("Its only OTHER (real) alternative row, if any:")
print(scored[(scored['Mfr Part #'] == self_ref_part) & (~scored['is_self_reference'])]
      [['Mfr Part #', 'Alternative Part', 'relationship', 'ASR']])


Part with the self-referencing row: MPC8270CZQMIBA
         Mfr Part #  best_ASR  n_real_alts  redundancy_credit  PARS PARS_category
140  MPC8270CZQMIBA      65.0          1.0                1.0  65.0          High

Its only OTHER (real) alternative row, if any:
         Mfr Part # Alternative Part relationship   ASR
162  MPC8270CZQMIBA   MPC8271CZQMIBA      Similar  65.0


In [11]:
# CELL 11: Validation 5 -- sanity check monotonicity: Identical-only parts should score lowest on average,
# No-Alternative parts should score highest, with Direct/Similar in between
summary = (
    part_scores.merge(
        real_alts.loc[real_alts.groupby('Mfr Part #')['ASR'].idxmin(), ['Mfr Part #', 'relationship']],
        on='Mfr Part #', how='left'
    )
    .assign(relationship=lambda d: d['relationship'].fillna('No real alternative'))
    .groupby('relationship')['PARS'].agg(['mean', 'count'])
    .reindex(['Identical', 'Direct', 'Similar', 'No real alternative'])
)
print(summary)
print("\nMonotonic increase Identical -> Direct -> Similar -> No real alternative:",
      summary['mean'].is_monotonic_increasing)


                          mean  count
relationship                         
Identical              4.79065    246
Direct                23.00000     25
Similar               64.28500    100
No real alternative  100.00000     62

Monotonic increase Identical -> Direct -> Similar -> No real alternative: True


## 6. Results Export


In [12]:
# CELL 12: Row-level and part-level outputs
row_output_cols = ['id', 'Mfr Part #', 'Alternative Part', 'manufacturer', 'relationship',
                    'is_no_alternative', 'is_self_reference', 'ASR', 'ASR_category']
row_level_output = scored[row_output_cols].sort_values('ASR', ascending=False)

part_level_output = part_scores[
    ['Mfr Part #', 'n_real_alts', 'best_ASR', 'redundancy_credit', 'PARS', 'PARS_category']
].sort_values('PARS', ascending=False)

row_level_output.to_csv('alternative_row_level_ASR.csv', index=False)
part_level_output.to_csv('part_level_PARS.csv', index=False)

print("Row-level ASR — worst 10:")
display(row_level_output.head(10))
print("\nPart-level PARS — worst 10:")
display(part_level_output.head(10))


Row-level ASR — worst 10:


,id,Mfr Part #,Alternative Part,manufacturer,relationship,is_no_alternative,is_self_reference,ASR,ASR_category
274,275,JLC1562BN,No Alternative,—,—,True,False,100.0,Critical
459,460,842S104EGLF,No Alternative,—,—,True,False,100.0,Critical
320,321,M61538FP#DF0G,No Alternative,—,—,True,False,100.0,Critical
429,430,SI53154A01AGMR,No Alternative,—,—,True,False,100.0,Critical
98,99,CM202001TR,No Alternative,—,—,True,False,100.0,Critical
323,324,LV3311PNMTLME,No Alternative,—,—,True,False,100.0,Critical
324,325,LV3328PMTLME,No Alternative,—,—,True,False,100.0,Critical
325,326,TAS5001IPFB,No Alternative,—,—,True,False,100.0,Critical
99,100,A1006UK/TA1NXZ,No Alternative,—,—,True,False,100.0,Critical
328,329,FAN3852UC16X,No Alternative,—,—,True,False,100.0,Critical



Part-level PARS — worst 10:


,Mfr Part #,n_real_alts,best_ASR,redundancy_credit,PARS,PARS_category
359,842S104EGLF,0.0,100.0,1.0,100.0,Critical
324,9FGV0241AKILFT,0.0,100.0,1.0,100.0,Critical
88,A1006TL/TA1NXZ,0.0,100.0,1.0,100.0,Critical
89,LC898229XIMH,0.0,100.0,1.0,100.0,Critical
50,CAP200DGTL,0.0,100.0,1.0,100.0,Critical
261,XVF3800QF60BC,0.0,100.0,1.0,100.0,Critical
325,LMKDB1202REYR,0.0,100.0,1.0,100.0,Critical
228,TCM320AC36CPT,0.0,100.0,1.0,100.0,Critical
91,CM2030A0TR,0.0,100.0,1.0,100.0,Critical
236,M62457AFP#TF0G,0.0,100.0,1.0,100.0,Critical


## 7. Summary

**Feature analysis:** of 7 raw columns, exactly **one** (`relationship`) carries independent
substitution-risk signal at the row grain; `manufacturer` and `description` are fully collinear
with it; `Last_Modified_Date` and `id` carry none; `Mfr Part #`/`Alternative Part` are structural
keys used for grouping and anomaly detection, not scoring inputs.

**Methodology review:** the reference notebook's ordinal ranking of `relationship`
(`N_AQR`) and its treat-zero-alternatives-as-worst-case logic (`N_SSE`) are directionally sound
and reused with independent justification; its manufacturer-diversity factor (`N_MDR`) doesn't
apply to a row-level target; and a plausible new feature (part-number string divergence) was
tested and rejected on direct empirical evidence rather than assumed to work.

**Final model:** `ASR` (row) is a 4-value lookup table anchored in IEC 62402 Form-Fit-Function
reasoning; `PARS` (part) takes the best documented alternative and applies a small, capped
multi-sourcing credit. No linear weighted sum, no arbitrary coefficients — every number in the
model (5 / 25 / 65 / 100, and the 1.00 / 0.90 / 0.80 credits) is individually justified in
Section 4, and Section 5 demonstrates the resulting scores behave monotonically and handle both
identified data-quality edge cases (missing alternatives, the self-referencing row) correctly.

**Honest limitations:**
- With only `relationship` as substantive signal, the row-level model is intentionally simple —
  adding unjustified complexity (e.g. the rejected string-divergence feature) would have reduced
  accuracy, not improved it, given what this specific table contains.
- The redundancy credit values (1.00/0.90/0.80) are a defensible, bounded, round-number scale
  consistent with ASCM multi-sourcing guidance, but — like any bounded discount scale — they are a
  judgment call rather than something estimable from this data alone; they are disclosed as such
  rather than presented as empirically derived.
- This score deliberately does **not** re-derive obsolescence, compliance, or documentation risk
  (`N_LCR`, `N_RSC`, `N_PKR`, etc.) — those live in the reference notebook and depend on tables not
  provided here. `PARS` is a *substitution-risk* score, one input a combined `Overall_Part_Risk`
  could draw on, not a replacement for that broader model.
